### Import Dependencies

In [3]:
import sys
sys.path.append("../")
import pandas as pd

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model,should_use_reasoning_model
from IPython.display import display, Markdown
from utils.token_utils import count_messages_tokens,fit_within_context
from utils.data_loader import load_incidents

In [4]:
path="../data/raw/Incidents.txt"
text  = load_incidents(path)
print(text)

ID: 1, Time: 08:00 AM, Area: Gampaha, People: 4, Ages: 20-40, Main Need: Water, Message: Thirsty but safe on roof. Water level stable.
ID: 2, Time: 08:15 AM, Area: Ja-Ela, People: 1, Ages: 75, Main Need: Insulin, Message: Diabetic, missed dose yesterday. Feeling faint.
ID: 3, Time: 08:20 AM, Area: Ragama, People: 2, Ages: 10, 35, Main Need: Rescue, Message: Water approaching neck level. Child is crying.


###  checking long chain-spam messages & apply the logic

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": text}
]

#checking If message > 150 tokens or not , if it's so truncate it
number_of_tokens = count_messages_tokens(
    messages=messages,
    provider="openai",
    model="gpt-3.5-turbo"
)

if(number_of_tokens['estimated_total'] > 150):
    # trunating
    messages = fit_within_context(
        messages=messages,
        provider="openai",
        model="gpt-3.5-turbo",
        max_context_tokens=150,
        strategy='truncate'
    )

    print(messages[0][1]['content'])
    #Print ”BLOCKED/TRUNCATED” for a spam input.
    print("\n\nBLOCKED/TRUNCATED")
else:
    #Print the response from the LLM
    print("Response from LLM")


    

ID: 1, Time: 08:00 AM, Area: Gampaha, People: 4, Ages: 20-40, Main Need: Water, Message: Thirsty but safe on roof. Water level stable.
ID: 2, Time: 08:15 AM, Area: Ja-Ela, People: 1, Ages: 75, Main Need: Insulin, Message: Diabetic, missed dose yesterday. Feeling faint.
ID: 3, Time: 08:20 AM, Area: Ragama, People: 2, Ages: 10, 35, Main Need: Rescue, Message: Water approaching neck level... [truncated]


BLOCKED/TRUNCATED
